<a href="https://colab.research.google.com/github/Maheesh09/AI-gateway/blob/main/SE3082_Lab09_IT24103477.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SE3082 – Lab 09: Introduction to CUDA (Google Colab Edition)

**Name:** Maheesha Pramuditha  
**Student ID:** IT24103477

## Exercise 1 – Set up your CUDA environment in Google Colab

In [1]:
!nvidia-smi

Sat Oct 10 19:30:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


**GPU:** Tesla T4  
**Compute capability:** 7.5  
**Compiler flag used in this notebook:** `-arch=sm_75`

In [4]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();          // 1 block of 4 threads
    cudaDeviceSynchronize();    // wait so the GPU's output is flushed
    return 0;
}

Writing hello.cu


In [5]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


## Exercise 2 – Simple calculation program from the lecture (adding two numbers)

In [6]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (the NVIDIA GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;                  // host copies of a, b, c
    int *d_a, *d_b, *d_c;         // device copies of a, b, c
    int size = sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Set up input values
    a = 2;
    b = 7;

    // 1. Copy the inputs from host to device
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // 2. Launch the add() kernel on the GPU: 1 block, 1 thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Colab edition: report a failed launch instead of silently printing garbage
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // 3. Copy the result from device to host
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex2_add.cu


In [8]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9


### Exercise 2 – THINK answers
1. **Why separate `d_a, d_b, d_c`?** The kernel runs on the GPU, and the GPU can only read its own memory. `a`, `b`, `c` live in the CPU's RAM, so the kernel cannot use them. `cudaMalloc` creates matching variables in GPU memory, and `cudaMemcpy` moves the values across.
2. **`&a` in `cudaMemcpy` but `d_a` in the launch?** `cudaMemcpy` needs the *addresses* of source and destination. `a` is a plain `int`, so its address is `&a`. `d_a` is already a pointer (an address in GPU memory), so it is passed as it is. The kernel needs GPU addresses, so it gets `d_a`; `&a` is a CPU address the GPU cannot use.
3. **How many threads?** `add<<<1, 1>>>` runs **1 thread** (1 block × 1 thread). For 10 threads use `add<<<1, 10>>>`. The answer stays **9**: every thread computes the same `2 + 7` and writes it to the same `*c`, so the extra threads only repeat the same work.

## Exercise 3 – Vector addition using 512 blocks and one thread

### Prediction
`add<<<N, 1>>>` launches **512 blocks** with **1 thread each = 512 threads in total**.
The GPU executes threads in **warps of 32**. A block with only one thread still occupies a whole warp, so 1 lane does work and **31 lanes sit idle**. 512 blocks × 32 lanes = 16,384 lanes for 512 useful threads, so about 97% of the warp capacity is wasted. The result is correct, but the launch is inefficient.

In [9]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU with N blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex3_blocks.cu


In [10]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
35 + 57 = 92
86 + 14 = 100
92 + 68 = 160
49 + 5 = 54
21 + 58 = 79
62 + 12 = 74
27 + 86 = 113
90 + 0 = 90
59 + 46 = 105
63 + 26 = 89
26 + 94 = 120
40 + 16 = 56
26 + 52 = 78
72 + 78 = 150
36 + 29 = 65
11 + 46 = 57
68 + 90 = 158
67 + 47 = 114
29 + 70 = 99
82 + 51 = 133
30 + 80 = 110
62 + 31 = 93
23 + 93 = 116
67 + 57 = 124
35 + 27 = 62
29 + 12 = 41
2 + 86 = 88
22 + 14 = 36
58 + 55 = 113
69 + 12 = 81
67 + 90 = 157
93 + 12 = 105
56 + 79 = 135
11 + 10 = 21
42 + 69 = 111
29 + 89 = 118
73 + 74 = 147
21 + 55 = 76
19 + 41 = 60
84 + 20 = 104
37 + 33 = 70
98 + 87 = 185
24 + 88 = 112
15 + 38 = 53
70 + 66 = 136
13 + 70 = 83
26 + 84 = 110
91 + 56 = 147
80 + 17 = 97
56 + 6 = 62
73 + 60 = 133
62 + 49 = 111
70 + 37 = 107
96 + 5 = 101
81 + 59 = 140
5 + 17 = 22
25 + 18 = 43
84 + 45 = 129
27 + 83 = 110
36 + 73 = 109
5 + 58 = 63
46 + 73 = 119
29 + 37 = 66
13 + 89 = 102
57 + 83 = 140
24 + 7 = 31
95 + 78 = 173
82 + 57 = 139
45 + 14 = 59
14 +

## Exercise 4 – Vector addition using one block and 512 threads

In [11]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex4_threads.cu


In [12]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

In [13]:
# Experiment: Exercise 4 with N = 2048 (built from a copy, so ex4_threads.cu keeps N = 512)
!sed 's/#define N 512/#define N 2048/' ex4_threads.cu > ex4_2048.cu
!nvcc -arch=sm_75 ex4_2048.cu -o ex4_2048 && ./ex4_2048 > ex4_2048.txt; head -n 3 ex4_2048.txt; tail -n 1 ex4_2048.txt

Kernel launch failed: invalid argument
0) 83 + 51 = 0
1) 86 + 80 = 0
Verification: 2048 mismatches out of 2048


### Experiment and THINK answers
**N = 2048:** the program prints `Kernel launch failed: invalid configuration argument`, then wrong results (the `c` values are 0 or garbage) and `Verification: 2048 mismatches out of 2048`. The `cudaGetLastError()` check is the part of the program that told us. The limit hit is the **maximum of 1024 threads per block**, so `<<<1, 2048>>>` is an invalid launch configuration and the kernel never ran.

**THINK:** one block cannot cover a very large array, because a block holds at most 1024 threads (and one block runs on a single multiprocessor, so it cannot use the whole GPU). The solution is many blocks of many threads, which is Exercise 5.

## Exercise 5 – Multiply two 10,000,000-element vectors using blocks and threads

In [14]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000              // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread (use blockIdx, blockDim, threadIdx)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array.
    // 19,532 x 512 = 10,000,384 threads but only 10,000,000 elements,
    // so the last 384 threads must not touch memory.
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: print the last 1000 results as "index) a x b = c"
    for (int i = N - 1000; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    // TODO 7: verify all N results on the CPU and print the number of mismatches
    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex5_vecmul.cu


In [15]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999001) 76 x 74 = 5624
9999002) 41 x 94 = 3854
9999003) 8 x 0 = 0
9999004) 1 x 41 = 41
9999005) 28 x 52 = 1456
9999006) 45 x 4 = 180
9999007) 63 x 35 = 2205
9999008) 89 x 97 = 8633
9999009) 10 x 71 = 710
9999010) 74 x 91 = 6734
9999011) 65 x 41 = 2665
9999012) 29 x 60 = 1740
9999013) 53 x 5 = 265
9999014) 35 x 88 = 3080
9999015) 18 x 12 = 216
9999016) 17 x 12 = 204
9999017) 8 x 65 = 520
9999018) 51 x 34 = 1734
9999019) 79 x 64 = 5056
9999020) 13 x 89 = 1157
9999021) 10 x 85 = 850
9999022) 7 x 2 = 14
9999023) 31 x 51 = 1581
9999024) 88 x 86 = 7568
9999025) 59 x 8 = 472
9999026) 94 x 50 = 4700
9999027) 58 x 94 = 5452
9999028) 5 x 53 = 265
9999029) 36 x 67 = 2412
9999030) 80 x 45 = 3600
9999031) 33 x 5 = 165
9999032) 64 x 41 = 2624
9999033) 21 x 91 = 1911
9999034) 93 x 6 = 558
9999035) 65 x 34 = 2210
9999036) 1 x 43 = 43
9999037) 90 x 62 = 5580
9999038) 80 x 21 = 1680
9999039) 90 x 92 = 8280
9999040) 1 x 86 = 86
9999041) 54 x 1

### THINK answer
Threads execute in **warps of 32**. 512 = 16 × 32, so every warp in a block is full and no lanes are wasted.

With **500** threads per block: blocks = (10,000,000 + 499) / 500 = **20,000** exactly, so the total is **10,000,000 threads** and there are no extra threads at the end. But each block still needs ⌈500 / 32⌉ = 16 warps = 512 lanes, so **12 lanes are idle in every block**: 20,000 × 12 = **240,000 wasted lanes**, compared with only 384 idle threads when using 512. A multiple of 32 keeps every warp full.

## Exercise 6 – Multiply two 10,000 × 10,000 matrices element by element

In [16]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]  (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    // TODO 1: row comes from the y dimension, col from the x dimension
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if BOTH row and col are inside the matrix
    // TODO 3: convert (row, col) into a flat index: row * cols + col
    if (row < rows && col < cols) {
        int idx = row * cols + col;          // max 99,999,999: fits in an int
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;       // device copies
    size_t count = (size_t)ROWS * COLS;
    size_t size  = count * sizeof(int);          // 400 MB per matrix

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);
    // TODO 4: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    // TODO 5: launch mulElem with <<<numBlocks, threadsPerBlock>>>
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 6: copy d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: print C[9999][9000] ... C[9999][9999]  (the last 1000 elements)
    int r = ROWS - 1;
    for (int col = COLS - 1000; col < COLS; col++) {
        size_t idx = (size_t)r * COLS + col;
        printf("C[%d][%d] = %d x %d = %d\n", r, col, a[idx], b[idx], c[idx]);
    }

    // TODO 8: verify all elements on the CPU and print the number of mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex6_matrix.cu


In [17]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

Grid: 313 x 625 blocks
C[9999][9000] = 29 x 56 = 1624
C[9999][9001] = 58 x 69 = 4002
C[9999][9002] = 76 x 50 = 3800
C[9999][9003] = 68 x 21 = 1428
C[9999][9004] = 67 x 78 = 5226
C[9999][9005] = 98 x 11 = 1078
C[9999][9006] = 3 x 29 = 87
C[9999][9007] = 69 x 52 = 3588
C[9999][9008] = 93 x 64 = 5952
C[9999][9009] = 60 x 65 = 3900
C[9999][9010] = 49 x 77 = 3773
C[9999][9011] = 5 x 44 = 220
C[9999][9012] = 6 x 77 = 462
C[9999][9013] = 17 x 31 = 527
C[9999][9014] = 45 x 65 = 2925
C[9999][9015] = 17 x 50 = 850
C[9999][9016] = 59 x 2 = 118
C[9999][9017] = 68 x 4 = 272
C[9999][9018] = 70 x 28 = 1960
C[9999][9019] = 54 x 11 = 594
C[9999][9020] = 7 x 87 = 609
C[9999][9021] = 29 x 93 = 2697
C[9999][9022] = 12 x 88 = 1056
C[9999][9023] = 14 x 26 = 364
C[9999][9024] = 74 x 6 = 444
C[9999][9025] = 24 x 52 = 1248
C[9999][9026] = 4 x 14 = 56
C[9999][9027] = 75 x 76 = 5700
C[9999][9028] = 0 x 40 = 0
C[9999][9029] = 65 x 7 = 455
C[9999][9030] = 67 x 13 = 871
C[9999][9031] = 29 x 96 = 2784
C[9999][9032] 